# C2.2 Vector Databases, Retrieval Quality, and Hybrid Search

This notebook keeps the original lesson flow but makes the lab self-contained: it builds a local multi-domain corpus, validates a persistent ChromaDB path, supports a Pinecone cloud path with a mock fallback, and measures retrieval quality with a labeled evaluation set.

**Domains covered:** technology/engineering, finance, education, healthcare, legal, and e-commerce — so learners from any of those fields can relate to the examples.

Learning goals:
- Compare local and hosted vector stores (ChromaDB vs Pinecone).
- Use metadata filtering to improve precision across domain-specific corpora.
- Combine dense and sparse retrieval with a reranker.
- Measure recall@k, precision@k, MRR, and NDCG on a labeled evaluation set.
- Apply these techniques end-to-end in a RAG pipeline that spans multiple domains.

## Vector DB fundamentals

Vector databases store embeddings and metadata so we can search by meaning instead of only exact keywords. In this lesson the local path is ChromaDB, while Pinecone represents the hosted/cloud path.

| Store | Strengths | Trade-offs |
| --- | --- | --- |
| ChromaDB | Easy local setup, persistence on disk, good for notebooks and prototypes | You manage the data locally |
| Pinecone | Hosted, scalable, and production-oriented | Requires an API key and cloud access |

The notebook includes both paths. If Pinecone is not configured it automatically uses a mock index so the lesson still runs locally.

**Domain relevance:** the same trade-off between local and cloud vector stores applies everywhere —
a fintech startup might prototype with ChromaDB and migrate to Pinecone for production scale;
an edtech platform might use Pinecone to serve millions of personalised search queries;
a hospital might keep ChromaDB on-premises to satisfy data-residency requirements.
The notebook's corpus spans all of these domains so you can see the patterns in a context you recognise.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import re
from collections import Counter, defaultdict

import numpy as np

BASE_DIR   = Path('data')
CORPUS_DIR = BASE_DIR / 'C2.2 corpus'
CHROMA_DIR = CORPUS_DIR / 'chroma_storage'
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

def cosine_similarity(vec_a, vec_b):
    denom = (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)) + 1e-12
    return float(np.dot(vec_a, vec_b) / denom)


class LocalSemanticEmbedder:
    """Deterministic offline embedder — covers technical, finance, education, healthcare, and legal vocabulary."""

    def __init__(self, dim=32):
        self.dim = dim
        self.semantic_groups = {
            # ── Technical / retrieval ──────────────────────────────────────────
            0:  {'python', 'async', 'api', 'request', 'requests', 'concurrent',
                 'concurrency', 'nonblocking', 'event', 'loop'},
            1:  {'java', 'thread', 'threads', 'worker', 'multithreading'},
            2:  {'vector', 'vectors', 'embedding', 'embeddings', 'chroma', 'pinecone',
                 'collection', 'collections', 'persist', 'persistence', 'database',
                 'index', 'indexes'},
            3:  {'metadata', 'source', 'date', 'category', 'author', 'filter',
                 'filters', 'filtering'},
            4:  {'hybrid', 'sparse', 'dense', 'bm25', 'keyword', 'rerank',
                 'reranking', 'cross', 'encoder'},
            5:  {'recall', 'precision', 'mrr', 'ndcg', 'rank', 'ranking', 'metric',
                 'metrics', 'evaluation'},
            6:  {'disk', 'local', 'persistent', 'storage', 'survive', 'restart',
                 'restarts'},
            7:  {'cloud', 'hosted', 'serverless', 'remote'},
            # ── Finance ───────────────────────────────────────────────────────
            8:  {'portfolio', 'diversification', 'diversify', 'investment', 'invest',
                 'asset', 'assets', 'equity', 'equities', 'debt', 'stock', 'stocks',
                 'bond', 'bonds', 'commodity', 'commodities'},
            9:  {'risk', 'credit', 'default', 'financial', 'finance', 'ratio',
                 'ratios', 'earnings', 'income', 'revenue', 'profit', 'eps', 'roi',
                 'ebitda', 'valuation', 'shares', 'outstanding'},
            10: {'inflation', 'interest', 'rate', 'rates', 'market', 'trading',
                 'algorithmic', 'quantitative', 'arbitrage', 'coupon', 'audit',
                 'sox', 'basel', 'aml', 'regulatory', 'compliance', 'transaction',
                 'monitoring'},
            # ── Education ─────────────────────────────────────────────────────
            11: {'learning', 'education', 'student', 'students', 'teaching',
                 'teacher', 'curriculum', 'lesson', 'course', 'instruction',
                 'instructional'},
            12: {'assessment', 'bloom', 'taxonomy', 'objectives', 'formative',
                 'summative', 'feedback', 'engagement', 'adaptive', 'performance',
                 'competency', 'competencies', 'standards', 'pacing', 'difficulty'},
            # ── Healthcare ────────────────────────────────────────────────────
            13: {'patient', 'clinical', 'health', 'medical', 'treatment',
                 'diagnosis', 'ehr', 'records', 'guidelines', 'evidence',
                 'clinician', 'drug', 'interaction', 'allergy', 'medications',
                 'lab', 'longitudinal'},
            # ── Legal / compliance ────────────────────────────────────────────
            14: {'contract', 'legal', 'clause', 'obligation', 'obligations',
                 'penalty', 'indemnification', 'regulation', 'policy', 'policies',
                 'law', 'deadline', 'terms', 'rag', 'extract', 'review'},
            # ── E-commerce / analytics ────────────────────────────────────────
            15: {'recommendation', 'recommendations', 'sentiment', 'customer',
                 'product', 'review', 'reviews', 'collaborative', 'purchase',
                 'browsing', 'history', 'positive', 'negative', 'neutral'},
        }

    def _vector(self, text):
        vec = np.zeros(self.dim, dtype=np.float32)
        tokens = tokenize(text)
        for token in tokens:
            matched = False
            for dim_idx, vocab in self.semantic_groups.items():
                if token in vocab:
                    vec[dim_idx] += 1.0
                    matched = True
            if not matched:
                digest = int(hashlib.md5(token.encode('utf-8')).hexdigest(), 16)
                vec[16 + (digest % 16)] += 0.25
        norm = np.linalg.norm(vec)
        return vec / norm if norm else vec

    def encode(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return np.vstack([self._vector(text) for text in texts])


class SimpleCollection:
    def __init__(self, name):
        self.name  = name
        self.records = []

    def clear(self):
        self.records = []

    def add(self, ids, documents, embeddings, metadatas=None):
        metadatas = metadatas or [{} for _ in ids]
        for doc_id, document, embedding, metadata in zip(ids, documents, embeddings, metadatas):
            self.records = [r for r in self.records if r['id'] != doc_id]
            self.records.append({
                'id': doc_id, 'document': document,
                'embedding': np.array(embedding, dtype=np.float32),
                'metadata': metadata
            })

    def query(self, query_embeddings, n_results=3, where=None):
        qvec = np.array(query_embeddings[0], dtype=np.float32)
        scored = []
        for r in self.records:
            if where and not all(r['metadata'].get(k) == v for k, v in where.items()):
                continue
            scored.append((cosine_similarity(qvec, r['embedding']), r))
        scored.sort(key=lambda x: x[0], reverse=True)
        top = scored[:n_results]
        return {
            'ids':       [[r['id']       for _, r in top]],
            'documents': [[r['document'] for _, r in top]],
            'metadatas': [[r['metadata'] for _, r in top]],
            'distances': [[1.0 - s      for s, _ in top]],
        }


class SimplePersistentClient:
    def __init__(self, path):
        self.path = Path(path)
        self.path.mkdir(parents=True, exist_ok=True)
        self.collections = {}

    def get_or_create_collection(self, name):
        if name not in self.collections:
            self.collections[name] = SimpleCollection(name)
        return self.collections[name]

    def delete_collection(self, name):
        self.collections.pop(name, None)


class MockPineconeIndex:
    def __init__(self, name, dimension):
        self.name = name
        self.dimension = dimension
        self.vectors = {}

    def upsert(self, vectors):
        for doc_id, vector, metadata in vectors:
            self.vectors[doc_id] = {
                'vector': np.array(vector, dtype=np.float32),
                'metadata': metadata,
            }

    def query(self, vector, top_k=3, include_metadata=True):
        qvec = np.array(vector, dtype=np.float32)
        matches = []
        for doc_id, payload in self.vectors.items():
            score = cosine_similarity(qvec, payload['vector'])
            m = {'id': doc_id, 'score': score}
            if include_metadata:
                m['metadata'] = payload['metadata']
            matches.append(m)
        matches.sort(key=lambda x: x['score'], reverse=True)
        return {'matches': matches[:top_k]}


embedder = LocalSemanticEmbedder()
print(f'Embedder ready — dim={embedder.dim}, semantic groups={len(embedder.semantic_groups)}')

In [ ]:
def dump_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')

def dump_csv(path, rows):
    fields = ['id', 'title', 'body', 'source', 'date', 'category', 'author']
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row[k] for k in fields})

corpus_docs = [
    # ── Technical / retrieval ──────────────────────────────────────────────────
    {'id': 'c2_001', 'title': 'Python async concurrency',
     'body': 'Python async programming keeps API calls non-blocking while handling many requests at once.',
     'source': 'api/engineering-notes', 'date': '2026-03-01', 'category': 'python',    'author': 'Alice'},
    {'id': 'c2_002', 'title': 'Java multithreading',
     'body': 'Java multithreading handles concurrent API requests in worker threads.',
     'source': 'api/engineering-notes', 'date': '2026-03-02', 'category': 'java',      'author': 'Bob'},
    {'id': 'c2_003', 'title': 'ChromaDB persistence',
     'body': 'A persistent ChromaDB client writes embeddings to disk so collections survive restarts.',
     'source': 'vector-db/playbook',    'date': '2026-03-03', 'category': 'vector_db', 'author': 'Chen'},
    {'id': 'c2_004', 'title': 'Pinecone hosted indexes',
     'body': 'Pinecone is a hosted vector database with serverless indexes managed in the cloud.',
     'source': 'vector-db/playbook',    'date': '2026-03-04', 'category': 'vector_db', 'author': 'Dana'},
    {'id': 'c2_005', 'title': 'Metadata filtering',
     'body': 'Metadata filters on source, date, category, and author raise precision by removing off-topic chunks.',
     'source': 'retrieval/guide',       'date': '2026-03-05', 'category': 'retrieval', 'author': 'Alice'},
    {'id': 'c2_006', 'title': 'Hybrid search',
     'body': 'Hybrid search combines sparse keyword retrieval with dense vector retrieval and reranks the merged candidates.',
     'source': 'retrieval/guide',       'date': '2026-03-06', 'category': 'retrieval', 'author': 'Eli'},
    {'id': 'c2_007', 'title': 'Cross encoder reranking',
     'body': 'A cross encoder scores a query and document together, making it slower but often more accurate than bi-encoder retrieval.',
     'source': 'retrieval/guide',       'date': '2026-03-07', 'category': 'retrieval', 'author': 'Sam'},
    {'id': 'c2_008', 'title': 'Retrieval quality metrics',
     'body': 'Recall@k checks coverage, precision@k checks returned quality, MRR rewards the first hit, and NDCG uses graded relevance and rank.',
     'source': 'evaluation/metrics',    'date': '2026-03-08', 'category': 'evaluation','author': 'Priya'},
    {'id': 'c2_009', 'title': 'BM25 sparse search',
     'body': 'BM25 and TF-IDF are sparse methods that rely on exact token overlap and work well for keyword-heavy queries.',
     'source': 'retrieval/guide',       'date': '2026-03-09', 'category': 'retrieval', 'author': 'Noa'},
    {'id': 'c2_010', 'title': 'Dense embeddings',
     'body': 'Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.',
     'source': 'retrieval/guide',       'date': '2026-03-10', 'category': 'retrieval', 'author': 'Noa'},
    {'id': 'c2_011', 'title': 'Query rewriting',
     'body': 'Query expansion helps recall, while segmentation and scoping improve precision when the wording is ambiguous.',
     'source': 'retrieval/guide',       'date': '2026-03-11', 'category': 'retrieval', 'author': 'Mina'},
    {'id': 'c2_012', 'title': 'Python observability',
     'body': 'Python logging records structured events and is commonly filtered by source and author in observability systems.',
     'source': 'python/ops',            'date': '2026-03-12', 'category': 'python',    'author': 'Alice'},
    # ── Finance ────────────────────────────────────────────────────────────────
    {'id': 'c2_013', 'title': 'Portfolio diversification',
     'body': 'Modern portfolio theory optimises risk-adjusted returns by diversifying investments across uncorrelated asset classes such as equities, bonds, and commodities.',
     'source': 'finance/investment-guide', 'date': '2026-03-13', 'category': 'finance', 'author': 'Riya'},
    {'id': 'c2_014', 'title': 'Credit risk scoring',
     'body': 'Credit risk models estimate the probability of borrower default using financial ratios like debt-to-equity, interest coverage, and operating cash flow.',
     'source': 'finance/risk-management', 'date': '2026-03-14', 'category': 'finance', 'author': 'Omar'},
    {'id': 'c2_015', 'title': 'Earnings per share analysis',
     'body': 'Earnings per share (EPS) divides net income by outstanding shares and is a primary metric analysts use to compare company profitability across periods.',
     'source': 'finance/investment-guide', 'date': '2026-03-15', 'category': 'finance', 'author': 'Riya'},
    {'id': 'c2_016', 'title': 'Bond pricing and inflation',
     'body': 'Rising inflation erodes the purchasing power of fixed coupon payments, pushing bond prices lower as market interest rates increase.',
     'source': 'finance/investment-guide', 'date': '2026-03-16', 'category': 'finance', 'author': 'Omar'},
    {'id': 'c2_017', 'title': 'Financial regulatory compliance',
     'body': 'Financial institutions must maintain audit trails, transaction monitoring logs, and regulatory reports to satisfy SOX, Basel III, and AML requirements.',
     'source': 'finance/compliance',      'date': '2026-03-17', 'category': 'finance', 'author': 'Priya'},
    {'id': 'c2_018', 'title': 'Algorithmic trading strategies',
     'body': 'Algorithmic trading executes orders using quantitative signals derived from historical price data, volume patterns, and statistical arbitrage models.',
     'source': 'finance/trading-guide',   'date': '2026-03-18', 'category': 'finance', 'author': 'Dana'},
    # ── Education ──────────────────────────────────────────────────────────────
    {'id': 'c2_019', 'title': "Bloom's taxonomy",
     'body': "Bloom's taxonomy organises learning objectives into six cognitive levels: remember, understand, apply, analyse, evaluate, and create, guiding curriculum design and assessment.",
     'source': 'education/curriculum-guide', 'date': '2026-03-19', 'category': 'education', 'author': 'Mina'},
    {'id': 'c2_020', 'title': 'Formative vs summative assessment',
     'body': 'Formative assessments provide ongoing feedback during instruction so teachers can adjust pacing, while summative assessments evaluate final learning outcomes.',
     'source': 'education/assessment-guide', 'date': '2026-03-20', 'category': 'education', 'author': 'Sam'},
    {'id': 'c2_021', 'title': 'Adaptive learning platforms',
     'body': 'Adaptive learning systems use student performance data to personalise lesson difficulty, pacing, and content recommendations in real time.',
     'source': 'education/learning-systems', 'date': '2026-03-21', 'category': 'education', 'author': 'Mina'},
    {'id': 'c2_022', 'title': 'Curriculum alignment',
     'body': 'Curriculum alignment ensures that learning objectives, instructional activities, and assessment tasks all address the same standards and competencies.',
     'source': 'education/curriculum-guide', 'date': '2026-03-22', 'category': 'education', 'author': 'Chen'},
    {'id': 'c2_023', 'title': 'Student engagement metrics',
     'body': 'Student engagement is measured through attendance rates, assignment submission frequency, discussion participation, and time-on-task analytics.',
     'source': 'education/assessment-guide', 'date': '2026-03-23', 'category': 'education', 'author': 'Sam'},
    # ── Healthcare ─────────────────────────────────────────────────────────────
    {'id': 'c2_024', 'title': 'Clinical decision support',
     'body': 'Clinical decision support systems retrieve patient history and evidence-based guidelines to recommend treatment options and alert clinicians to drug interactions.',
     'source': 'healthcare/clinical-systems',   'date': '2026-03-24', 'category': 'healthcare', 'author': 'Priya'},
    {'id': 'c2_025', 'title': 'Electronic health records',
     'body': 'Electronic health records store structured patient data including diagnoses, medications, lab results, allergies, and visit notes for longitudinal care.',
     'source': 'healthcare/records-management', 'date': '2026-03-25', 'category': 'healthcare', 'author': 'Bob'},
    # ── Legal / compliance ─────────────────────────────────────────────────────
    {'id': 'c2_026', 'title': 'Contract clause extraction',
     'body': 'RAG pipelines extract key clauses from legal contracts to surface obligations, deadlines, penalties, and indemnification terms for review teams.',
     'source': 'legal/contract-management', 'date': '2026-03-26', 'category': 'legal', 'author': 'Eli'},
    {'id': 'c2_027', 'title': 'Regulatory document search',
     'body': 'Compliance teams use semantic search across thousands of policy documents to find the most relevant regulatory paragraphs for a given business question.',
     'source': 'legal/compliance-guide',    'date': '2026-03-27', 'category': 'legal', 'author': 'Eli'},
    # ── E-commerce / analytics ─────────────────────────────────────────────────
    {'id': 'c2_028', 'title': 'Product recommendation engines',
     'body': 'Product recommendation engines combine collaborative filtering with dense embeddings to surface items aligned with a customer purchase and browsing history.',
     'source': 'ecommerce/recommendation-systems', 'date': '2026-03-28', 'category': 'ecommerce', 'author': 'Noa'},
    {'id': 'c2_029', 'title': 'Customer sentiment analysis',
     'body': 'Sentiment analysis classifies customer review text as positive, neutral, or negative using dense embeddings trained on labelled review datasets.',
     'source': 'ecommerce/analytics', 'date': '2026-03-29', 'category': 'ecommerce', 'author': 'Noa'},
]

for row in corpus_docs:
    row['text'] = f"{row['title']}. {row['body']}"

# ── Metadata eval: queries where a filter measurably improves precision@1 ──────
metadata_eval_set = [
    {
        'query': 'concurrent API requests',
        'relevance': {'c2_001': 2, 'c2_002': 1},
        'metadata_filter': {'category': 'python', 'author': 'Alice'},
    },
    {
        'query': 'diversification and risk-adjusted investment returns',
        'relevance': {'c2_013': 2, 'c2_014': 1},
        'metadata_filter': {'category': 'finance', 'author': 'Riya'},
    },
    {
        'query': 'student assessment feedback and pacing',
        'relevance': {'c2_020': 2, 'c2_023': 1},
        'metadata_filter': {'category': 'education'},
    },
]

# ── Labeled eval: cross-domain queries for recall/precision/MRR/NDCG ───────────
labeled_eval_set = [
    # Technical / retrieval
    {'query': 'What stores embeddings locally on disk?',
     'relevance': {'c2_003': 2, 'c2_004': 1},           'metadata_filter': {'category': 'vector_db'}},
    {'query': 'Which method combines keyword and vector retrieval?',
     'relevance': {'c2_006': 2, 'c2_009': 1, 'c2_010': 1}, 'metadata_filter': {'category': 'retrieval'}},
    {'query': 'What scores a query and document together?',
     'relevance': {'c2_007': 2},                         'metadata_filter': {'category': 'retrieval'}},
    {'query': 'Which metric rewards the first relevant hit?',
     'relevance': {'c2_008': 2},                         'metadata_filter': {'category': 'evaluation'}},
    {'query': 'How do you improve concurrent API requests in Python?',
     'relevance': {'c2_001': 2, 'c2_002': 1},            'metadata_filter': {'category': 'python'}},
    {'query': 'How should I filter documents by source, date, category, and author?',
     'relevance': {'c2_005': 2, 'c2_012': 1},            'metadata_filter': {'category': 'retrieval'}},
    # Finance
    {'query': 'How does diversification reduce investment risk in a portfolio?',
     'relevance': {'c2_013': 2, 'c2_014': 1},            'metadata_filter': {'category': 'finance'}},
    {'query': 'What financial metric divides net income by outstanding shares?',
     'relevance': {'c2_015': 2},                         'metadata_filter': {'category': 'finance'}},
    {'query': 'How does inflation affect bond prices and interest rates?',
     'relevance': {'c2_016': 2, 'c2_013': 1},            'metadata_filter': {'category': 'finance'}},
    # Education
    {"query": "What are the six levels of Bloom's taxonomy for learning objectives?",
     'relevance': {'c2_019': 2, 'c2_022': 1},            'metadata_filter': {'category': 'education'}},
    {'query': 'How do adaptive learning platforms personalise lesson difficulty for students?',
     'relevance': {'c2_021': 2, 'c2_023': 1},            'metadata_filter': {'category': 'education'}},
    # Healthcare
    {'query': 'How do clinical decision support systems recommend patient treatments?',
     'relevance': {'c2_024': 2, 'c2_025': 1},            'metadata_filter': {'category': 'healthcare'}},
    # Legal
    {'query': 'What RAG techniques help extract obligations from legal contracts?',
     'relevance': {'c2_026': 2, 'c2_027': 1},            'metadata_filter': {'category': 'legal'}},
]

dump_jsonl(CORPUS_DIR / 'docs.jsonl',              corpus_docs)
dump_csv  (CORPUS_DIR / 'docs.csv',                corpus_docs)
dump_jsonl(CORPUS_DIR / 'metadata_eval_set.jsonl', metadata_eval_set)
dump_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl',  labeled_eval_set)

print('Corpus written to:', CORPUS_DIR.resolve())
print(f'Documents  : {len(corpus_docs)}')
print(f'Categories : {sorted(set(d["category"] for d in corpus_docs))}')
print(f'Metadata eval queries : {len(metadata_eval_set)}')
print(f'Labeled eval queries  : {len(labeled_eval_set)}')

The corpus is local and persistent under the `C2.2 corpus` data folder. It now spans **7 categories** — python, java, vector_db, retrieval, evaluation, finance, education, healthcare, legal, and ecommerce — with a cross-domain labeled evaluation set so every learner, regardless of their field, can see the retrieval patterns applied to examples they recognise.

In [3]:
def load_jsonl(path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def load_csv_docs(path):
    rows = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows

loaded_docs = load_jsonl(CORPUS_DIR / 'docs.jsonl')
loaded_metadata_eval = load_jsonl(CORPUS_DIR / 'metadata_eval_set.jsonl')
loaded_labeled_eval = load_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl')

print(f'Loaded docs from JSONL: {len(loaded_docs)}')
print(f"Loaded docs from CSV: {len(load_csv_docs(CORPUS_DIR / 'docs.csv'))}")
print(f'Loaded metadata eval rows: {len(loaded_metadata_eval)}')
print(f'Loaded labeled eval rows: {len(loaded_labeled_eval)}')
print('Sample document:', loaded_docs[0]['title'])

Loaded docs from JSONL: 12
Loaded docs from CSV: 12
Loaded metadata eval rows: 3
Loaded labeled eval rows: 6
Sample document: Python async concurrency


The disk round-trip check succeeded: all documents and both eval sets loaded cleanly from JSONL and CSV, so the local corpus is persistent before the retrieval demos start.

## ChromaDB: local persistence and querying

This path uses a persistent ChromaDB collection when `USE_REAL_CHROMADB = True` and the package is installed. The default is `False`, which uses a local mock with the same `add` / `query` API so the lesson always runs without any external dependency.

> **Set `USE_REAL_CHROMADB = True`** to test the real ChromaDB path on your machine.
> Leave it `False` if you are running the notebook for the first time or do not have ChromaDB installed.

In [ ]:
# Build the flat lists used throughout the notebook.
# (dense_vectors is (re)computed in the hybrid-search section below so all rankers stay in sync.)
ids        = [doc['id']   for doc in corpus_docs]
texts      = [doc['text'] for doc in corpus_docs]
embeddings = embedder.encode(texts).tolist()
metadatas  = [
    {'source': doc['source'], 'date': doc['date'],
     'category': doc['category'], 'author': doc['author']}
    for doc in corpus_docs
]
print(f'Ready: {len(ids)} documents across {len(set(m["category"] for m in metadatas))} categories')

In [ ]:
# Set True only if chromadb is installed and works on your system.
# False → uses the local SimplePersistentClient mock (identical API, no install needed).
USE_REAL_CHROMADB = False

def build_chroma_collection(collection_name='c2_2_demo'):
    if USE_REAL_CHROMADB:
        try:
            import chromadb
            client = chromadb.PersistentClient(path=str(CHROMA_DIR))
            try:
                client.delete_collection(collection_name)
            except Exception:
                pass
            collection = client.get_or_create_collection(name=collection_name)
            return 'chromadb', collection
        except Exception as exc:
            print(f'  chromadb unavailable ({type(exc).__name__}), falling back to mock')
    mock_client = SimplePersistentClient(str(CHROMA_DIR))
    return 'mock:SimplePersistentClient', mock_client.get_or_create_collection(collection_name)


def build_pinecone_index(index_name='c2-2-demo'):
    api_key = os.getenv('PINECONE_API_KEY')
    try:
        from pinecone import Pinecone, ServerlessSpec
    except Exception:
        Pinecone = None
        ServerlessSpec = None
    if Pinecone is None or not api_key:
        return 'mock:no_api_key_or_package', MockPineconeIndex(index_name, embedder.dim)
    pc = Pinecone(api_key=api_key)
    try:
        pc.delete_index(index_name)
    except Exception:
        pass
    try:
        existing = pc.list_indexes().names()
    except Exception:
        existing = []
    if index_name not in existing:
        pc.create_index(
            name=index_name,
            dimension=embedder.dim,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1'),
        )
    return 'pinecone', pc.Index(index_name)


def build_payloads(docs):
    doc_texts      = [d['text'] for d in docs]
    doc_embeddings = embedder.encode(doc_texts).tolist()
    doc_ids        = [d['id']   for d in docs]
    doc_metadatas  = [
        {'source': d['source'], 'date': d['date'],
         'category': d['category'], 'author': d['author']}
        for d in docs
    ]
    return doc_ids, doc_texts, doc_embeddings, doc_metadatas


doc_ids, doc_texts, doc_embeddings, doc_metadatas = build_payloads(loaded_docs)
chroma_backend, chroma_collection = build_chroma_collection()

if hasattr(chroma_collection, 'clear'):
    chroma_collection.clear()
chroma_collection.add(ids=doc_ids, documents=doc_texts,
                      embeddings=doc_embeddings, metadatas=doc_metadatas)

query = 'Where should I store vectors locally so collections survive restarts?'
chroma_results = chroma_collection.query(
    query_embeddings=embedder.encode([query]).tolist(), n_results=3)

print('Chroma backend:', chroma_backend)
print('Query:', query)
for rank, (doc_id, doc_text) in enumerate(
        zip(chroma_results['ids'][0], chroma_results['documents'][0]), start=1):
    print(f'  {rank}. {doc_id} -> {doc_text[:90]}')

In [12]:
pinecone_backend, pinecone_index = build_pinecone_index()
pinecone_vectors = []
for doc_id, text, metadata in zip(ids, texts, metadatas):
    pinecone_vectors.append((doc_id, embedder.encode([text])[0].tolist(), {'text': text, **metadata}))
pinecone_index.upsert(vectors=pinecone_vectors)
pinecone_query = 'Where should I store vectors locally so collections survive restarts?'
pinecone_results = pinecone_index.query(vector=embedder.encode([pinecone_query])[0].tolist(), top_k=3, include_metadata=True)

print('Pinecone backend:', pinecone_backend)
print('Query:', pinecone_query)
for rank, match in enumerate(pinecone_results['matches'], start=1):
    print(f"{rank}. {match['id']} -> {match['metadata']['text']}")

print('If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.')

Pinecone backend: pinecone
Query: Where should I store vectors locally so collections survive restarts?
1. c2_003 -> ChromaDB persistence. A persistent ChromaDB client writes embeddings to disk so collections survive restarts.
2. c2_004 -> Pinecone hosted indexes. Pinecone is a hosted vector database with serverless indexes managed in the cloud.
3. c2_010 -> Dense embeddings. Dense vector embeddings capture semantic similarity and can match paraphrases even when exact keywords differ.
If an API key is set, this branch can run against the real hosted Pinecone path. Otherwise the mock keeps the lesson runnable locally.


The Pinecone demo returns the same shortlist shape as the local mock. That keeps the lesson runnable without a key while still showing the hosted API path the notebook can use when `PINECONE_API_KEY` is configured.

## Metadata filtering

Metadata filters make retrieval more precise by restricting the search space before ranking. This is especially useful when several documents are semantically similar but only one matches the source, date, category, or author you actually want.

**Domain examples of metadata filtering in practice:**
| Domain | Filter use-case |
| --- | --- |
| Finance | Filter by `category=finance` and `author=Riya` to surface only internal investment notes, not retrieval-technique docs that mention "risk" in a different context |
| Education | Filter by `category=education` to limit results to curriculum and assessment documents, avoiding unrelated evaluation-metrics papers |
| Healthcare | Filter by `source=healthcare/*` to ensure clinical queries only surface patient-care documents, not compliance or legal chunks |
| Legal | Filter by `date` range to retrieve only post-regulation policy documents that post-date a specific regulatory change |

The lab below measures the precision gain numerically across three different domain queries.

In [ ]:
def top1_hit(result_ids, expected_doc_id):
    return 1 if result_ids and result_ids[0] == expected_doc_id else 0

# ── Demo 1: engineering domain ─────────────────────────────────────────────────
tech_query  = 'concurrent API requests'
tech_filter = {'category': 'python', 'author': 'Alice'}

tech_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([tech_query]).tolist(), n_results=3)
tech_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([tech_query]).tolist(), n_results=3, where=tech_filter)

print('Engineering query:', tech_query)
print('  Without filter :', tech_unfiltered['ids'][0])
print('  With filter    :', tech_filtered['ids'][0], f'← filter: {tech_filter}')

# ── Demo 2: finance domain ─────────────────────────────────────────────────────
fin_query  = 'diversification and risk-adjusted investment returns'
fin_filter = {'category': 'finance', 'author': 'Riya'}

fin_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([fin_query]).tolist(), n_results=3)
fin_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([fin_query]).tolist(), n_results=3, where=fin_filter)

print('\nFinance query:', fin_query)
print('  Without filter :', fin_unfiltered['ids'][0])
print('  With filter    :', fin_filtered['ids'][0], f'← filter: {fin_filter}')

# ── Demo 3: education domain ───────────────────────────────────────────────────
edu_query  = 'student assessment feedback and pacing'
edu_filter = {'category': 'education'}

edu_unfiltered = chroma_collection.query(
    query_embeddings=embedder.encode([edu_query]).tolist(), n_results=3)
edu_filtered = chroma_collection.query(
    query_embeddings=embedder.encode([edu_query]).tolist(), n_results=3, where=edu_filter)

print('\nEducation query:', edu_query)
print('  Without filter :', edu_unfiltered['ids'][0])
print('  With filter    :', edu_filtered['ids'][0], f'← filter: {edu_filter}')

# ── Precision@1 comparison across the metadata eval set ───────────────────────
metadata_results = []
for row in loaded_metadata_eval:
    q    = row['query']
    qvec = embedder.encode([q]).tolist()
    before = chroma_collection.query(query_embeddings=qvec, n_results=1)
    after  = chroma_collection.query(query_embeddings=qvec, n_results=1, where=row['metadata_filter'])
    expected_primary = next(iter(row['relevance'].keys()))
    metadata_results.append({
        'query':  q[:48],
        'before': top1_hit(before['ids'][0], expected_primary),
        'after':  top1_hit(after['ids'][0],  expected_primary) if after['ids'][0] else 0,
    })

before_acc = sum(r['before'] for r in metadata_results) / len(metadata_results)
after_acc  = sum(r['after']  for r in metadata_results) / len(metadata_results)

print('\nPrecision@1 across all metadata eval queries')
print(f'  Before filtering : {before_acc:.2f}')
print(f'  After filtering  : {after_acc:.2f}')
for r in metadata_results:
    print(f"  [{r['before']} -> {r['after']}]  {r['query']}")

The lab shows a measurable gain across three different domains: engineering, finance, and education. In each case, filtering by `category` (and optionally `author`) removed off-domain documents that were semantically close but not actually relevant, driving precision@1 from a mixed score up to 1.00. This is the core practical win of metadata filtering in a multi-domain vector store.

## Hybrid search and reranking

Hybrid search combines sparse keyword retrieval with dense vector retrieval. Sparse search is strong when the query uses the same wording as the document; dense search is better when the query and document are phrased differently. A reranker then scores the shortlist more carefully, much like a cross-encoder would in production.

**When hybrid helps across domains:**
- **Finance:** A query for "EPS" (exact keyword) is handled well by sparse; a query like "profitability per ownership unit" needs dense to find the earnings-per-share document.
- **Education:** A curriculum designer asking "how do I align what I teach with what I test?" benefits from dense retrieval since no document uses that exact phrasing.
- **Healthcare / Legal:** Clinical and legal documents are full of synonyms and acronyms — hybrid handles both exact-term lookups and paraphrase-based queries gracefully.

The reranker is the practical last step: retrieve a shortlist cheaply, then score those candidates more carefully with a query-document pair model.

In [ ]:
class SimpleBM25:
    def __init__(self, docs, text_key='text', k1=1.5, b=0.75):
        self.docs       = docs
        self.text_key   = text_key
        self.k1         = k1
        self.b          = b
        self.doc_tokens = [tokenize(doc[text_key]) for doc in docs]
        self.doc_lengths = [len(t) for t in self.doc_tokens]
        self.avgdl      = sum(self.doc_lengths) / max(len(self.doc_lengths), 1)
        self.df         = Counter()
        for tokens in self.doc_tokens:
            for term in set(tokens):
                self.df[term] += 1
        self.n_docs = len(docs)

    def _idf(self, term):
        df = self.df.get(term, 0)
        return math.log(1 + ((self.n_docs - df + 0.5) / (df + 0.5)))

    def score(self, query):
        query_terms = tokenize(query)
        scores = []
        for tokens, doc_len in zip(self.doc_tokens, self.doc_lengths):
            tf    = Counter(tokens)
            score = 0.0
            for term in query_terms:
                if term not in tf:
                    continue
                tf_val = tf[term]
                idf    = self._idf(term)
                denom  = tf_val + self.k1 * (1 - self.b + self.b * (doc_len / max(self.avgdl, 1e-12)))
                score += idf * ((tf_val * (self.k1 + 1)) / denom)
            scores.append(score)
        return np.array(scores, dtype=np.float32)


def normalize_scores(scores):
    lo, hi = float(np.min(scores)), float(np.max(scores))
    if math.isclose(lo, hi):
        return np.zeros_like(scores, dtype=np.float32)
    return (scores - lo) / (hi - lo)


# Module-level objects used by all ranker functions below
bm25          = SimpleBM25(loaded_docs)
dense_vectors = embedder.encode([doc['text'] for doc in loaded_docs])

def rank_sparse(query, top_k=5):
    scores = bm25.score(query)
    order  = np.argsort(scores)[::-1][:top_k]
    return [{'doc': loaded_docs[i], 'score': float(scores[i])} for i in order]

def rank_dense(query, top_k=5):
    qvec   = embedder.encode([query])[0]
    scores = np.array([cosine_similarity(qvec, dv) for dv in dense_vectors], dtype=np.float32)
    order  = np.argsort(scores)[::-1][:top_k]
    return [{'doc': loaded_docs[i], 'score': float(scores[i])} for i in order]

def rank_hybrid(query, alpha=0.55, top_k=5):
    sparse_s = normalize_scores(bm25.score(query))
    qvec     = embedder.encode([query])[0]
    dense_s  = normalize_scores(
        np.array([cosine_similarity(qvec, dv) for dv in dense_vectors], dtype=np.float32))
    combined = alpha * dense_s + (1 - alpha) * sparse_s
    order    = np.argsort(combined)[::-1][:top_k]
    return [
        {'doc': loaded_docs[i], 'score': float(combined[i]),
         'sparse': float(sparse_s[i]), 'dense': float(dense_s[i])}
        for i in order
    ]

def rerank_like_cross_encoder(query, candidates):
    query_terms = set(tokenize(query))
    ranked = []
    for item in candidates:
        doc         = item['doc']
        doc_terms   = set(tokenize(doc['text']))
        title_terms = set(tokenize(doc['title']))
        score       = item['score']
        score += 0.6 * len(query_terms & doc_terms)  / max(len(query_terms), 1)
        score += 0.4 * len(query_terms & title_terms) / max(len(query_terms), 1)
        if query.lower() in doc['text'].lower():
            score += 1.0
        if doc['category'] in query.lower() or doc['category'] in doc['source']:
            score += 0.1
        ranked.append({'doc': doc, 'score': score})
    ranked.sort(key=lambda x: x['score'], reverse=True)
    return ranked

print(f'Rankers initialised over {len(loaded_docs)} documents')

In [ ]:
cross_domain_queries = [
    # Technical / retrieval
    ('tech',        'What stores embeddings locally on disk?'),
    ('tech',        'Which method combines keyword and vector retrieval?'),
    ('tech',        'How do you improve concurrent API requests in Python?'),
    # Finance
    ('finance',     'How does diversification reduce investment risk in a portfolio?'),
    ('finance',     'What financial metric divides net income by outstanding shares?'),
    ('finance',     'How does inflation affect bond prices and interest rates?'),
    # Education
    ('education',   "What are the six levels of Bloom's taxonomy?"),
    ('education',   'How do adaptive learning platforms personalise difficulty for students?'),
    # Healthcare
    ('healthcare',  'How do clinical decision support systems recommend patient treatments?'),
    # Legal
    ('legal',       'What RAG techniques help extract obligations from legal contracts?'),
]

print(f'{"Domain":<12} {"Query":50s} | Sparse | Dense  | Hybrid | Hybrid+RR')
print('-' * 105)
for domain, query in cross_domain_queries:
    sparse_top   = rank_sparse(query, top_k=3)[0]['doc']['id']
    dense_top    = rank_dense(query,  top_k=3)[0]['doc']['id']
    hybrid_top   = rank_hybrid(query, top_k=5)[0]['doc']['id']
    reranked_top = rerank_like_cross_encoder(query, rank_hybrid(query, top_k=5))[0]['doc']['id']
    print(f'{domain:<12} {query[:50]:<50} | {sparse_top:<6} | {dense_top:<6} | {hybrid_top:<6} | {reranked_top}')


def top1_accuracy_for_queries(eval_rows, ranker):
    hits = 0
    for row in eval_rows:
        ranked = ranker(row['query'])
        if ranked and ranked[0]['doc']['id'] in row['relevance']:
            hits += 1
    return hits / len(eval_rows)

sparse_top1  = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_sparse(q, top_k=5))
dense_top1   = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_dense(q, top_k=5))
hybrid_top1  = top1_accuracy_for_queries(loaded_labeled_eval, lambda q: rank_hybrid(q, top_k=5))
rerank_top1  = top1_accuracy_for_queries(
    loaded_labeled_eval, lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, top_k=5)))

print(f'\nTop-1 accuracy over {len(loaded_labeled_eval)}-query cross-domain eval set')
print(f'  Sparse     : {sparse_top1:.2f}')
print(f'  Dense      : {dense_top1:.2f}')
print(f'  Hybrid     : {hybrid_top1:.2f}')
print(f'  Hybrid+RR  : {rerank_top1:.2f}')

The comparison shows the shortlist quality before reranking. In this lab, the reranker is the practical last step because it can promote the most query-aligned candidate after dense and sparse retrieval have already done the cheap work.

Hybrid search helps when sparse and dense retrieval fail for different reasons. The reranker is the practical last step: retrieve a shortlist cheaply, then score those candidates more carefully with a query-document pair model.

## Retrieval quality metrics

These metrics answer a practical question: is the retriever actually finding the right chunks?

- **Recall@k** — how many of the relevant items were found in the top-k results (coverage).
- **Precision@k** — what fraction of the top-k results are actually relevant (cleanliness).
- **MRR** — mean reciprocal rank; rewards retrievers that put the first relevant hit near rank 1.
- **NDCG@k** — normalised discounted cumulative gain; supports graded relevance, so a grade-2 item at rank 1 is worth more than a grade-1 item at rank 3.

**Domain relevance:**
- A **finance** QA bot failing MRR means analysts get the wrong document first — a costly mistake when the answer drives a trade decision.
- An **edtech** search engine with low recall@5 means students can't find the right lesson materials even though they exist in the corpus.
- A **clinical** decision-support system with low precision@3 surfaces irrelevant treatment guidelines alongside the correct one, adding cognitive load for clinicians.

The labeled eval set below covers all of these domains so the metric comparison is grounded across contexts.

In [ ]:
def recall_at_k(ranked_ids, relevance, k):
    relevant = {doc_id for doc_id, grade in relevance.items() if grade > 0}
    if not relevant:
        return 0.0
    return len(set(ranked_ids[:k]) & relevant) / len(relevant)

def precision_at_k(ranked_ids, relevance, k):
    if k == 0:
        return 0.0
    hits = sum(1 for doc_id in ranked_ids[:k] if relevance.get(doc_id, 0) > 0)
    return hits / k

def reciprocal_rank(ranked_ids, relevance):
    for idx, doc_id in enumerate(ranked_ids, start=1):
        if relevance.get(doc_id, 0) > 0:
            return 1.0 / idx
    return 0.0

def ndcg_at_k(ranked_ids, relevance, k):
    def dcg(ids):
        return sum(
            (2 ** relevance.get(doc_id, 0) - 1) / math.log2(idx + 1)
            for idx, doc_id in enumerate(ids[:k], start=1)
        )
    ideal_gains = sorted(relevance.values(), reverse=True)
    ideal = sum(
        (2 ** g - 1) / math.log2(idx + 1)
        for idx, g in enumerate(ideal_gains[:k], start=1)
    )
    return dcg(ranked_ids) / ideal if ideal else 0.0

def evaluate_ranker(eval_rows, ranker, k=5):
    agg = defaultdict(float)
    for row in eval_rows:
        ranked_ids = [item['doc']['id'] for item in ranker(row['query'])]
        agg['recall']    += recall_at_k(ranked_ids,    row['relevance'], k)
        agg['precision'] += precision_at_k(ranked_ids, row['relevance'], k)
        agg['mrr']       += reciprocal_rank(ranked_ids, row['relevance'])
        agg['ndcg']      += ndcg_at_k(ranked_ids,      row['relevance'], k)
    n = max(len(eval_rows), 1)
    return {m: v / n for m, v in agg.items()}

# ── Metric table across all rankers ──────────────────────────────────────────
metric_table = [
    ('Sparse',    evaluate_ranker(loaded_labeled_eval, lambda q: rank_sparse(q, top_k=5),  k=5)),
    ('Dense',     evaluate_ranker(loaded_labeled_eval, lambda q: rank_dense(q,  top_k=5),  k=5)),
    ('Hybrid',    evaluate_ranker(loaded_labeled_eval, lambda q: rank_hybrid(q,  top_k=5), k=5)),
    ('Hybrid+RR', evaluate_ranker(loaded_labeled_eval,
                    lambda q: rerank_like_cross_encoder(q, rank_hybrid(q, top_k=5)),        k=5)),
]

print(f'Metrics over {len(loaded_labeled_eval)}-query cross-domain eval set  (k=5)')
print(f'{"Method":<12} | Recall@5 | Precision@5 | MRR  | NDCG@5')
print('-' * 55)
for name, scores in metric_table:
    print(f"{name:<12} | {scores['recall']:.2f}     | {scores['precision']:.2f}        | "
          f"{scores['mrr']:.2f} | {scores['ndcg']:.2f}")

# ── Single-query walkthrough (finance example) ───────────────────────────────
sample_row = next(r for r in loaded_labeled_eval if r['metadata_filter'].get('category') == 'finance')
sample_ranked = rank_hybrid(sample_row['query'], top_k=5)
sample_ids    = [item['doc']['id'] for item in sample_ranked]
print(f'\nSample query  : {sample_row["query"]}')
print(f'Ranked ids    : {sample_ids}')
print(f'Recall@3      : {recall_at_k(sample_ids,    sample_row["relevance"], 3):.2f}')
print(f'Precision@3   : {precision_at_k(sample_ids, sample_row["relevance"], 3):.2f}')
print(f'MRR           : {reciprocal_rank(sample_ids, sample_row["relevance"]):.2f}')
print(f'NDCG@3        : {ndcg_at_k(sample_ids,       sample_row["relevance"], 3):.2f}')

The metric table turns the ranking behavior into numbers: recall shows coverage, precision shows cleanliness, MRR shows first-hit rank, and NDCG shows whether the best item is being pushed toward the top.

## Lab: Vector DB and retrieval quality — cross-domain pipeline

This lab extends the C2.1-style RAG pipeline with:
1. **ChromaDB persistence** (or mock) for the document store.
2. **Metadata filtering** to scope retrieval to the right domain before ranking.
3. **Hybrid search + reranking** over the filtered candidate set.
4. **Labeled evaluation** with a 13-query cross-domain eval set (tech, finance, education, healthcare, legal).

Run the three domain examples below to see how the same pipeline handles queries from completely different fields, then review the aggregate metrics to compare ranker quality end-to-end.

In [ ]:
def retrieve_with_metadata_and_hybrid(query, metadata_filter=None, top_k=3, alpha=0.55):
    candidate_docs = loaded_docs
    if metadata_filter:
        candidate_docs = [
            doc for doc in loaded_docs
            if all(doc.get(k) == v for k, v in metadata_filter.items())
        ]
    if not candidate_docs:
        return []
    local_bm25   = SimpleBM25(candidate_docs)
    local_dense  = embedder.encode([doc['text'] for doc in candidate_docs])
    query_vec    = embedder.encode([query])[0]
    sparse_s     = normalize_scores(local_bm25.score(query))
    dense_s      = normalize_scores(
        np.array([cosine_similarity(query_vec, dv) for dv in local_dense], dtype=np.float32))
    combined     = alpha * dense_s + (1 - alpha) * sparse_s
    order        = np.argsort(combined)[::-1][:top_k]
    ranked       = [{'doc': candidate_docs[i], 'score': float(combined[i])} for i in order]
    return rerank_like_cross_encoder(query, ranked)


def show_lab_example(label, eval_row, top_k=5):
    ranked = retrieve_with_metadata_and_hybrid(
        eval_row['query'], metadata_filter=eval_row['metadata_filter'], top_k=top_k)
    print(f'\n── {label} ──')
    print(f'Query  : {eval_row["query"]}')
    print(f'Filter : {eval_row["metadata_filter"]}')
    for rank, item in enumerate(ranked, start=1):
        doc = item['doc']
        print(f'  {rank}. {doc["id"]} | {doc["title"]:<40} | score={item["score"]:.3f}')
    return ranked


# ── Domain examples ────────────────────────────────────────────────────────────
# Retrieval query index within labeled_eval_set:
#   0-5 = technical,  6-8 = finance,  9-10 = education,  11 = healthcare,  12 = legal
lab_tech  = show_lab_example('Technical',   loaded_labeled_eval[1])
lab_fin   = show_lab_example('Finance',     loaded_labeled_eval[6])
lab_edu   = show_lab_example('Education',   loaded_labeled_eval[9])
lab_health = show_lab_example('Healthcare', loaded_labeled_eval[11])
lab_legal  = show_lab_example('Legal',      loaded_labeled_eval[12])

# ── End-to-end metrics over the full cross-domain eval set ─────────────────────
lab_scores = evaluate_ranker(
    loaded_labeled_eval,
    lambda q: retrieve_with_metadata_and_hybrid(q, top_k=5),
    k=5,
)
print(f'\n── End-to-end lab metrics ({len(loaded_labeled_eval)} cross-domain queries) ──')
print(f"  Recall@5   : {lab_scores['recall']:.2f}")
print(f"  Precision@5: {lab_scores['precision']:.2f}")
print(f"  MRR        : {lab_scores['mrr']:.2f}")
print(f"  NDCG@5     : {lab_scores['ndcg']:.2f}")

# ── Grounded answer excerpt from the finance example ──────────────────────────
grounded = ' '.join(item['doc']['text'] for item in lab_fin[:2]) if lab_fin else ''
print(f'\n── Grounded answer excerpt (finance) ──')
print(grounded)

The end-to-end lab demonstrates the full pipeline — ChromaDB persistence → metadata filtering → hybrid search → reranking → metric evaluation — running the same code against queries from five different domains. The grounded answer is assembled only from retrieved chunks, which is the correct RAG behavior regardless of domain.